# Agentic AI and Spatial Grounding

This notebook picks up where `TrafficSafety.ipynb` leaves off: it reads the
published Frankfurt traffic safety **risk layer**, retrieves the highest
confidence locations, and generates ranked, explainable recommendations
using `traffic_safety.recommendations`.

Set `RISK_LAYER_ITEM_ID` below to the item id returned by
`publish_hotspot_layer(...)` in the previous session.

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
import os

from arcgis.gis import GIS

from traffic_safety.recommendations import (
    TemplateLLMClient,
    get_risk_layer,
    get_top_risk_locations,
    rank_recommendations,
)

In [ ]:
home_portal = GIS("home")
risk_layer_item_id = os.getenv("RISK_LAYER_ITEM_ID")
risk_layer = get_risk_layer(home_portal, risk_layer_item_id)

## Top risk locations

Query the risk layer for the locations with the highest, most persistent
confidence (`Gi_Bin_median`).

In [ ]:
top_locations = get_top_risk_locations(risk_layer, top_n=5)
top_locations[["OBJECTID", "Gi_Bin_median", "Gi_Bin_years_observed", "Gi_Bin_min", "Gi_Bin_max"]]

## Generate explainable recommendations

`rank_recommendations` builds an evidence package per location and passes it
to an `LLMClient`. By default this uses the dependency-free
`TemplateLLMClient`, so the demo runs without any external AI service
configured. Swap in `traffic_safety.governance.OllamaLLMClient` (next
session) or any hosted-model client implementing the same `generate(prompt)`
interface to produce richer, free-form text from the exact same evidence.

In [ ]:
recommendations = rank_recommendations(risk_layer, top_n=5, llm_client=TemplateLLMClient())
for recommendation in recommendations:
    print(recommendation["recommendation"])
    print("-" * 80)

## Next steps

Each `recommendation` dict is a self-contained evidence package ready for
human review. Continue to `TrustedGeoAI.ipynb` to see these recommendations
go through a human-in-the-loop approval workflow with a full audit trail,
and to swap in a local/sovereign model via `OllamaLLMClient`.